# <span style="color:red; font-size:25px"> DATASCI 151 - Quiz #5 </span>

<font size = "4" >
This quiz is open book 

<br>

- You <span style="color:red"> CAN </span> use all material from the course repo

- You should <span style="color:red"> NOT </span> use other resources outside the course repo.

- You should <span style="color:red"> NOT </span> collaborate with other students

<!-- - This is a graded quiz. Use of LLMs or any AI tools is <span style="color:red"> NOT ALLOWED. </span> -->

- AI assistance is <span style="color:red"> NOT </span> permitted for this assignment.

- To get full credit, the code should work as intended. But try to attempt every problem - you can still receive partial credit.

<br>

Print the following string:

"I will abide by Emory's code of conduct"

In [ ]:
print("I will abide by EMory's code of conduct")

<font size = "4">

Import any packages you would like in the cell below. Or, import them as needed later in the notebook.

In [ ]:
# import packages
import pandas as pd
import numpy as np 

<font size = "4">

**Q1** Recode a numeric column

<font size = "3">

- Read in the file "movies.csv" from the `quiz_data` folder, saving it as a DataFrame.

- Recode the "year" variable to create a new variable called "time_period" with the following categories. (**Both** endpoints are included in the listed categories)

    - **Silent Era**: Films made between 1917 and 1927

    - **Golden Age**: Films made between 1928 and 1949

    - **Post-War**: Films made between 1950 and 1964

    - **New Hollywood**: Films made between 1965 and 1974

    - **Blockbuster Era**: Movies made between 1975 and 2000

    - **Digital Age**: Movies made between 2001 until the present

- You must use the `pd.cut` function to recode this variable.

- Add a new column to the DataFrame called "Era" that contains this recoded variable.



In [ ]:
df_movies = pd.read_csv("quiz_data/movies.csv")

year_bins = [1916, 1927, 1949, 1964, 1974, 2000, np.inf]
year_labels = ["Silent Era", "Golden Age", "Post-War", "New Hollywood", "Blockbuster Era", "Digital Age"]

df_movies["Era"] = pd.cut(x = df_movies["Year"],
                                  bins=year_bins,
                                  labels=year_labels
                                    )

display(df_movies.sort_values(by="Year")) ## sort by era or by year check

<font size = "4">

**Q2** Conditional Subsetting

<font size = "3">

- Continue to use the movies DataFrame from Q1 (or read it in again)

- Return a DataFrame with the **6 shortest movies** that (a) were released 2011 or later, and (b) have IMDb rating of 6.0 or greater.

**Hint**: The column containing the IMDb rating is named "IMDb rating", which would not be a valid Python variable because of the space. If this causes you trouble and you don't remember how to handle it, you could simply rename the column to "IMDb_rating".


In [ ]:
df_filter = df_movies[(df_movies["Year"] >= 2011) + (df_movies["IMDb rating"] >= 6.0)]
df_results = df_filter.sort_values(by="Duration (minutes)").head(6)
df_results

<font size = "4">

**Q3** Aggregating Group Statistics with Chaining

<font size = "3">

- Continue to use the movies DataFrame from Q1 (or read it in again)

- The column "MPA rating" shows whether the movie was rated R, PG-13, etc. Aggregate the following statistics for each MPA rating:

    - Mean IMDb rating

    - Median duration (in minutes)

- Sort the resulting DataFrame by mean IMDb rating in **ascending** order.

- For full credit, execute all commands on a single line by chaining



In [ ]:
df_agg = df_movies.groupby("MPA rating").agg(mean_IMDb = ('IMDb rating', 'mean'), median_duration = ('Duration (minutes)', 'median')).sort_values(by="mean_IMDb")
df_agg

<font size = "4">

**Q4** Calculate laps per minute

<font size = "3">

- Read in the file "results_cleaned.csv" from the `quiz_data` folder, saving it as a DataFrame.

- The "milliseconds" column shows the race duration for each result measured in milliseconds. Create a new column called "race_duration_minutes" where we convert the race duration to minutes by dividing each millisecond value by 60,000 (which equals $6 \times 10^4$). Equivalently, you can multiply each millisecond value by $6 \times 10^{-4}$

- Create a new column called "laps_per_minute" which is calculated by the formula: $\ \dfrac{\textrm{laps}}{\textrm{race\_duration\_minutes}}$

- Group the data by **both** "raceId" and "driverId" and calculate the **maximum laps per minutes** for each group.

- Sort the data in descending order and then extract the raceId/driverId pairs with the 5 highest laps per minute

<br>

**Hint**: For a Pandas Series, the `.sort_values()` method does **not** take the keyword "by".

In [ ]:
df_cleaned = pd.read_csv("quiz_data/results_cleaned.csv")
df_cleaned["race_duration_minutes"] = df_cleaned["milliseconds"] * 6e-4
df_cleaned["laps_per_minute"] = df_cleaned["laps"]/df_cleaned["race_duration_minbutes"]
gb = df_cleaned.groupby(["raceId","driverId"])["laps_per_minute"].max().sort_values(ascending=False).head(5)


<font size = "4">

**Q5** Aggregate and query

<font size = "3">

- Continue to use the "results_cleaned.csv" DataFrame from Q4 (or read it in again)

- Do the following in a single line using chaining:

    - Extract the subset with raceId equal to 21.

    - Aggregate the minimum number of points and average number of points for each constructor in the subset.

    - Extract the subset of these constructors with average number of points **strictly greater than 1** 

In [ ]:
df_cleaned[df_cleaned["raceId"]==21].groupby("constructorId").agg(min_points = ('points', 'min'), avg_points = ('points', 'mean')).query("avg_points > 1")

<font size = "4">

**Q6** Aggregate custom statistic

<font size = "3">

- Continue to use the DataFrame corresponding to the data in "results_cleaned.csv" (or read it in again).

- We might consider it a "bad day" for an F1 driver if they finish outside the top ten (11th or worse). Write a function that takes in a Pandas Series and returns the number of entries $ > 10$.

- Group by "driverId" and then aggregate the following statistics based on the "positionOrder" column: the average (mean) position and the number of finishes outside the top ten.

- Use your custom function for this second statistic.

- Merge the aggregated data back **into the original DataFrame**.

In [ ]:
# your code here
def bad_days(col):
    return (col > 10).sum()

df_agg = df_cleaned.groupby("driverId").agg(avg_position = ("positionOrder", "mean"),
                                             bad_days = ("positionOrder", bad_days))

df_cleaned = pd.merge(left=df_cleaned, right=df_agg, on="driverId", how="left")

df_cleaned